# 01 · Prep — F1 panels → one tabular `Xy`

Turns every F1 unit's time-series panel into rows of

> *(unit, asset, origin date, horizon)* → features known at the origin → the value h business days later

**Output:** `f1_pipeline/data/f1_Xy.parquet` and `f1_columns.json`.
No model is fitted here.

Run order: **01 → 02 → 03 → 04**.

## Setup

In [1]:
import json, pathlib, tomllib, time
import numpy as np, pandas as pd

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "f1_pipeline" else pathlib.Path.cwd()
OUT  = REPO / "f1_pipeline" / "data"; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
print("repo:", REPO)

repo: /Users/dew/track2-forecasting-public


## 1 — Load the units

One card + one panel per unit. The panel is pivoted wide (date × series); the target asset is one of
its columns.

In [2]:
def load_unit(unit_dir: pathlib.Path) -> dict:
    card = tomllib.loads((unit_dir / "card.toml").read_text())
    pq   = sorted(unit_dir.glob("*.parquet"))[0]
    df   = pd.read_parquet(pq)
    acol = "asset" if "asset" in df.columns else "asset_id"
    df["date"] = pd.to_datetime(df["date"].astype(str).str[:10])
    wide = df.pivot(index="date", columns=acol, values="value").sort_index().astype(float)
    t = card["targets"]
    return dict(unit=card["task"]["id"], panel=pq.stem, wide=wide,
                asof=pd.Timestamp(card["provenance"]["data_cutoff"]),
                assets=list(t["asset_ids"]), horizons=[int(h) for h in t["horizons"]],
                target_type=t["target_type"], value_unit=t["value_unit"],
                freq="monthly" if pq.stem == "macro_monthly" else "daily")

UNITS = [load_unit(p) for p in sorted(REPO.glob("units/t2-F1-*"))]
pd.DataFrame([{"unit": u["unit"], "panel": u["panel"], "freq": u["freq"], "as-of": u["asof"].date(),
               "targets": ",".join(u["assets"]), "horizons": u["horizons"], "target_type": u["target_type"],
               "panel_rows": len(u["wide"]), "series": u["wide"].shape[1]} for u in UNITS])

,unit,panel,freq,as-of,targets,horizons,target_type,panel_rows,series
0,t2-F1-ai-mom-2024,factors_daily,daily,2024-05-31,MOM,[127],log_return,6370,6
1,t2-F1-aud-on-hold-2016,g10_fx_daily,daily,2016-11-01,AUD,"[126, 189]",level,4232,10
2,t2-F1-cad-boc-2017,g10_fx_daily,daily,2017-07-12,CAD,"[126, 189]",level,4404,10
3,t2-F1-chf-highly-valued-2021,g10_fx_daily,daily,2021-06-17,CHF,"[126, 189]",level,5384,10
4,t2-F1-conflicting-texts-2024,rates_daily,daily,2024-06-12,UST_10Y,"[126, 189]",level,6116,6
5,t2-F1-considerable-period-2003,rates_daily,daily,2003-08-12,UST_10Y,"[126, 189]",level,903,6
6,t2-F1-conundrum-2005,rates_daily,daily,2005-02-18,UST_10Y,"[126, 189]",level,1283,6
7,t2-F1-cpi-glidepath-2023,macro_monthly,monthly,2023-07-12,CPI_ALL,"[140, 160]",level,281,6
8,t2-F1-dkk-peg-2019,g10_fx_daily,daily,2019-03-01,DKK,[129],level,4809,10
9,t2-F1-eur-range-2017,g10_fx_daily,daily,2017-09-07,EUR,"[126, 189]",level,4444,10


## 2 — Definitions

| term | meaning |
|---|---|
| **origin** *t* | a panel date we pretend is today; features use data ≤ *t* only |
| **horizon** *h* | business days ahead, from the card |
| **steps_ahead** *k* | rows ahead the target sits: *k = h* on daily panels; on monthly panels, months from the last panel row to the month of (as-of + *h* BD) |
| **working series** *x* | `level` → the level itself · `log_return` → cumulative log-return index, so *x*(t+k) − *x*(t) is the card's cumulative return |
| **anchor** | *x*(t) for level targets, 0 for log-return targets |
| **target** | the value in the card's unit |
| **target_change** | `target − anchor` — what every model actually predicts |
| **split** | `train` if the outcome is inside the panel · `predict` if *t* is the as-of row |

In [3]:
def working_series(wide, asset, target_type):
    s = wide[asset].dropna()                       # the asset's own trading days (panels can be ragged)
    return np.log1p(s).cumsum() if target_type == "log_return" else s

## 3 — Features

Window names are in business days; on monthly panels they convert to months (21 BD → 1 month), so one
schema covers both frequencies.

**Per-series** (from the target's own history): `level`, `log_level`, `mom_21/63/126/252`, `z_252`,
`pos_252`, `rv_21/63/252`, `ewma_vol`, `vol_ratio`, `skew_252`, `last_step`.

**Panel context** (`ctx_*`, from the other series in the same panel): curve slope/curvature for rates,
broad-USD strength for FX, equity trend/vol for factors, inflation and labour for macro. NaN where the
panel doesn't have them — that is "not applicable", not missing.

In [4]:
WINDOWS = {"21": 21, "63": 63, "126": 126, "252": 252}

def series_features(x: pd.Series, freq: str, is_level: bool) -> pd.DataFrame:
    scale = 21 if freq == "monthly" else 1
    w = {k: max(1, v // scale) for k, v in WINDOWS.items()}
    d = x.diff(); F = pd.DataFrame(index=x.index)
    F["level"]     = x if is_level else np.nan
    F["log_level"] = np.log(x.clip(lower=1e-3)) if is_level else np.nan
    for k in ("21", "63", "126", "252"): F[f"mom_{k}"] = x - x.shift(w[k])
    F["z_252"]   = (x - x.rolling(w["252"]).mean()) / x.rolling(w["252"]).std()
    F["pos_252"] = (x - x.rolling(w["252"]).min()) / (x.rolling(w["252"]).max() - x.rolling(w["252"]).min())
    for k in ("21", "63", "252"): F[f"rv_{k}"] = d.rolling(w[k]).std()
    F["ewma_vol"]  = d.ewm(alpha=1 - 0.94 ** scale).std()
    F["vol_ratio"] = F["rv_21"] / F["rv_252"]
    F["skew_252"]  = d.rolling(w["252"]).skew()
    F["last_step"] = d
    return F

USD_PER_CCY = {"AUD", "EUR", "GBP", "NZD"}    # quoted USD-per-unit; the other six are units-per-USD

def context_features(wide: pd.DataFrame, panel: str, freq: str) -> pd.DataFrame:
    C = pd.DataFrame(index=wide.index); m63 = 63 if freq == "daily" else 3
    if panel == "rates_daily":
        C["ctx_slope_10_2"]  = wide["UST_10Y"] - wide["UST_2Y"]
        C["ctx_slope_5_2"]   = wide["UST_5Y"]  - wide["UST_2Y"]
        C["ctx_curv_2_5_10"] = 2 * wide["UST_5Y"] - wide["UST_2Y"] - wide["UST_10Y"]
        C["ctx_ust2y"], C["ctx_ust10y"] = wide["UST_2Y"], wide["UST_10Y"]
        C["ctx_slope_mom_63"] = C["ctx_slope_10_2"].diff(m63)
    elif panel == "g10_fx_daily":
        lg = np.log(wide).apply(lambda col: -col if col.name in USD_PER_CCY else col)
        C["ctx_usd_mom_63"] = lg.diff(m63).mean(axis=1)
        C["ctx_usd_rv_63"]  = lg.diff().mean(axis=1).rolling(m63).std()
    elif panel == "factors_daily":
        mkt, mom = np.log1p(wide["MKT"].dropna()), np.log1p(wide["MOM"].dropna())
        C["ctx_mkt_mom_63"] = mkt.cumsum().diff(m63)
        C["ctx_mkt_rv_63"]  = mkt.rolling(m63).std()
        C["ctx_mom_mom_63"] = mom.cumsum().diff(m63)
    elif panel == "macro_monthly":
        C["ctx_cpi_yoy"]  = wide["CPI_ALL"].pct_change(12) * 100
        C["ctx_core_yoy"] = wide["CPI_CORE"].pct_change(12) * 100
        C["ctx_unrate"]   = wide["UNRATE"]
        C["ctx_unrate_chg_3"] = wide["UNRATE"].diff(3)
        C["ctx_nfp_3m"]   = wide["NFP"].diff(3)
    return C

## 4 — Build

In [5]:
def steps_ahead_for(u, h):
    if u["freq"] == "daily": return h
    target_date = u["asof"] + pd.offsets.BDay(h); last = u["wide"].index[-1]
    return (target_date.year - last.year) * 12 + (target_date.month - last.month)

def build_unit_rows(u: dict) -> pd.DataFrame:
    is_level = u["target_type"] == "level"
    ctx = context_features(u["wide"], u["panel"], u["freq"]); out = []
    for asset in u["assets"]:
        x = working_series(u["wide"], asset, u["target_type"])
        F = series_features(x, u["freq"], is_level); n, idx = len(x), x.index
        for h in u["horizons"]:
            k = steps_ahead_for(u, h)
            anchor = x.to_numpy() if is_level else np.zeros(n)
            tgt   = np.full(n, np.nan)
            tdate = np.full(n, np.datetime64("NaT", "ns"))
            tgt[:n - k]   = x.to_numpy()[k:] if is_level else (x.to_numpy()[k:] - x.to_numpy()[:n - k])
            tdate[:n - k] = idx[k:].to_numpy()
            D = pd.DataFrame({"unit": u["unit"], "panel": u["panel"], "freq": u["freq"], "asset": asset,
                              "target_type": u["target_type"], "value_unit": u["value_unit"], "asof": u["asof"],
                              "origin_date": idx, "origin_idx": np.arange(n), "horizon_bd": h, "steps_ahead": k,
                              "target_date": tdate, "anchor": anchor, "target": tgt}, index=idx)
            D["target_change"] = D["target"] - D["anchor"]
            is_asof = D["origin_date"] == idx[-1]
            D["split"] = np.where(D["target"].notna(), "train", np.where(is_asof, "predict", "drop"))
            out.append(pd.concat([D, F, ctx.reindex(idx).ffill()], axis=1).loc[D["split"] != "drop"])
    return pd.concat(out, ignore_index=True)

ID_COLS     = ["unit", "panel", "freq", "asset", "target_type", "value_unit", "asof",
               "origin_date", "origin_idx", "horizon_bd", "steps_ahead", "target_date", "split"]
TARGET_COLS = ["anchor", "target", "target_change"]

t0 = time.time()
Xy = pd.concat([build_unit_rows(u) for u in UNITS], ignore_index=True)
m = (Xy.split == "predict") & Xy.target_date.isna()          # monthly as-of rows have no panel date to point at
Xy.loc[m, "target_date"] = [a + pd.offsets.BDay(int(h)) for a, h in zip(Xy.loc[m, "asof"], Xy.loc[m, "horizon_bd"])]
FEATURE_COLS = [c for c in Xy.columns if c not in ID_COLS + TARGET_COLS]
print(f"{len(Xy):,} rows × {Xy.shape[1]} cols in {time.time()-t0:.1f}s  "
      f"({len(ID_COLS)} id, {len(TARGET_COLS)} target, {len(FEATURE_COLS)} feature)")
Xy.groupby(["freq", "target_type", "split"]).size().rename("rows").to_frame()

150,066 rows × 47 cols in 0.2s  (13 id, 3 target, 31 feature)


rows
freq    target_type split          
daily   level       predict      37
                    train    138135
        log_return  predict       2
                    train     10772
monthly level       predict       4
                    train      1116

## 5 — Checks

In [6]:
# (a) exactly one predict row per (unit, asset, horizon), on the unit's last panel date
pred = Xy[Xy.split == "predict"]
assert pred.groupby(["unit", "asset", "horizon_bd"]).size().eq(1).all()
assert (pred.origin_date == pred.groupby("unit").origin_date.transform("max")).all()
print(f"predict rows: {len(pred)}")

# (b) a level target reconstructed from the raw panel
u = next(u for u in UNITS if u["unit"] == "t2-F1-hawkish-cut-2024"); s = u["wide"]["UST_2Y"]
i = s.index.get_loc(pd.Timestamp("2019-10-30"))
row = Xy[(Xy.unit == u["unit"]) & (Xy.origin_date == "2019-10-30") & (Xy.horizon_bd == 126)].iloc[0]
assert np.isclose(row.target, s.iloc[i + 126]) and row.target_date == s.index[i + 126]
print(f"level  : panel {s.iloc[i+126]:.2f} on {s.index[i+126].date()}  ==  table {row.target:.2f} on {row.target_date.date()}")

# (c) a log-return target against the repo's own step definition
from qfbench2_track_forecasting.targets import log_return_steps
u = next(u for u in UNITS if u["unit"] == "t2-F1-ai-mom-2024"); r = u["wide"]["MOM"].dropna()
j = r.index.get_loc(pd.Timestamp("2020-03-02")); direct = np.log1p(r.iloc[j+1:j+1+127]).sum()
row = Xy[(Xy.unit == u["unit"]) & (Xy.origin_date == "2020-03-02")].iloc[0]
assert np.isclose(row.target, direct) and np.isclose(pd.Series(log_return_steps(r)).iloc[j+1:j+1+127].sum(), direct)
print(f"logret : direct {direct:+.4f}  ==  table {row.target:+.4f}   (matches log_return_steps)")

# (d) no look-ahead: features are identical if the panel is truncated at the origin
u_full = next(u for u in UNITS if u["unit"] == "t2-F1-hawkish-cut-2024")
u_cut = dict(u_full, wide=u_full["wide"].loc[:"2015-06-30"], asof=pd.Timestamp("2015-06-30"))
a = build_unit_rows(u_full).set_index(["origin_date", "horizon_bd"])
b = build_unit_rows(u_cut).set_index(["origin_date", "horizon_bd"])
common = b.index[b.index.get_level_values(0) <= "2015-06-30"]
fc = [c for c in FEATURE_COLS if c in a.columns]
print(f"look-ahead check: max |Δfeature| = {(a.loc[common, fc] - b.loc[common, fc]).abs().max().max():.1e}")
assert (a.loc[common, fc] - b.loc[common, fc]).abs().max().max() < 1e-9

predict rows: 43
level  : panel 0.19 on 2020-05-04  ==  table 0.19 on 2020-05-04


logret : direct -0.0730  ==  table -0.0730   (matches log_return_steps)
look-ahead check: max |Δfeature| = 0.0e+00


## 6 — Save

In [7]:
Xy.to_parquet(OUT / "f1_Xy.parquet", index=False)
(OUT / "f1_columns.json").write_text(json.dumps({"id": ID_COLS, "target": TARGET_COLS, "feature": FEATURE_COLS}, indent=2))
for p in sorted(OUT.glob("f1_*")): print(f"{p.name:<26} {p.stat().st_size/1e6:7.2f} MB")

f1_Xy.parquet                11.11 MB
f1_columns.json               0.00 MB


Next: **02 · Split** — one file per (unit, asset, horizon) plus the feature mapper.